In [ ]:
# Task 2.1: Arabic Text Preprocessing

import re
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

def normalize(text):
    if text is None:
        return ""

    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)

    return text

normalize_udf = udf(normalize, StringType())

df_norm = classification_df.withColumn(
    "normalized_text",
    normalize_udf(col("text"))
)

In [ ]:
# Task 2.1: Arabic Text Preprocessing

def remove_diacritics(text):
    if text is None:
        return ""

    text = re.sub(r'[\u0617-\u061A\u064B-\u0652]', '', text)
    return text

diacritic_udf = udf(remove_diacritics, StringType())

df_no_diac = df_norm.withColumn(
    "clean_text",
    diacritic_udf(col("normalized_text"))
)


In [ ]:
# Task 2.1: Arabic Text Preprocessing

import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
from pyspark.ml.feature import Tokenizer, StopWordsRemover

arabic_stopwords = stopwords.words('arabic')


tokenizer = Tokenizer(inputCol="clean_text", outputCol="words")
df_tokens = tokenizer.transform(df_no_diac)


remover = StopWordsRemover(
    inputCol="words",
    outputCol="filtered_words",
    stopWords=arabic_stopwords
)

df_stop = remover.transform(df_tokens)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# Task 2.1: Arabic Text Preprocessing

from pyspark.sql.types import ArrayType

def stem(words):
    return [w[:-1] if len(w) > 3 else w for w in words]

stem_udf = udf(stem, ArrayType(StringType()))

df_stem = df_stop.withColumn(
    "stemmed_words",
    stem_udf(col("filtered_words"))
)

In [ ]:
# Task 2.1: Arabic Text Preprocessing

from pyspark.sql.functions import concat_ws

df_final = df_stem.withColumn(
    "abstract_clean",
    concat_ws(" ", col("stemmed_words"))
)

In [ ]:
# Task 3.1: Traditional Features:

import re
from pyspark.sql.functions import udf, col
from pyspark.sql.types import IntegerType

# Feature 1 Number of words with repeated letters

def count_repeated_letters(text):
    if text is None:
        return 0

    count = 0
    words = text.split()

    for word in words:
        if re.search(r'(.)\1+', word):
            count += 1

    return count

repeated_letters_udf = udf(count_repeated_letters, IntegerType())

In [ ]:
# Task 3.1: Traditional Features:

# Feature 2 Total number of sentences

def count_sentences(text):
    if text is None or text.strip() == "":
        return 0

    sentences = re.split(r'[.!؟]+', text)
    sentences = [s for s in sentences if s.strip()]

    return len(sentences)

sentences_udf = udf(count_sentences, IntegerType())


In [ ]:
# Task 3.1: Traditional Features:

df_features = df_final.withColumn(
    "f016_words_with_repeated_letters",
    repeated_letters_udf(col("abstract_clean"))
)

df_features = df_features.withColumn(
    "f034_total_sentences",
    sentences_udf(col("text"))
)

df_features.limit(10).toPandas()

df_features.write.mode("overwrite").parquet("data/features/features_data.parquet")


In [ ]:
# Task 3.1: Traditional Features:

from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(
    inputCol="abstract_clean",
    outputCol="words_tfidf"
)

df_words = tokenizer.transform(df_features)

In [ ]:
# Task 3.2: Advanced Features

from pyspark.ml.feature import CountVectorizer

vectorizer = CountVectorizer(
    inputCol="words_tfidf",
    outputCol="tf_features"
)

cv_model = vectorizer.fit(df_words)
df_tf = cv_model.transform(df_words)


In [ ]:
# Task 3.2: Advanced Features

from pyspark.ml.feature import IDF

idf = IDF(
    inputCol="tf_features",
    outputCol="tfidf_features"
)

idf_model = idf.fit(df_tf)
df_tfidf = idf_model.transform(df_tf)

df_tfidf.limit(10).toPandas()

,text,label,normalized_text,clean_text,words,filtered_words,stemmed_words,abstract_clean,f016_words_with_repeated_letters,f034_total_sentences,words_tfidf,tf_features,tfidf_features
0,تعتبر اللّغة من أهمّ العوامل الّتي تعمل على إر...,0,تعتبر اللّغه من اهمّ العوامل الّتي تعمل علي ار...,تعتبر اللغه من اهم العوامل التي تعمل علي ارساء...,"[تعتبر, اللغه, من, اهم, العوامل, التي, تعمل, ع...","[تعتبر, اللغه, اهم, العوامل, تعمل, علي, ارساء,...","[تعتب, اللغ, اهم, العوام, تعم, علي, ارسا, روح,...",تعتب اللغ اهم العوام تعم علي ارسا روح الوحد ال...,5,1,"[تعتب, اللغ, اهم, العوام, تعم, علي, ارسا, روح,...","(3.0, 4.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.34682917097167654, 0.658595339289239, 0.0, ..."
1,تناص نظرية جديدة في الدراسات النقدية الحديثة، ...,0,تناص نظريه جديده في الدراسات النقديه الحديثه، ...,تناص نظريه جديده في الدراسات النقديه الحديثه، ...,"[تناص, نظريه, جديده, في, الدراسات, النقديه, ال...","[تناص, نظريه, جديده, الدراسات, النقديه, الحديث...","[تنا, نظري, جديد, الدراسا, النقدي, الحديثه, مص...",تنا نظري جديد الدراسا النقدي الحديثه مصطل طورت...,2,3,"[تنا, نظري, جديد, الدراسا, النقدي, الحديثه, مص...","(1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, ...","(0.11560972365722551, 0.0, 0.0, 0.0, 0.0, 0.77..."
2,يتناول المقال النص الشعري الجزائري المعاصر بال...,0,يتناول المقال النص الشعري الجزايري المعاصر بال...,يتناول المقال النص الشعري الجزايري المعاصر بال...,"[يتناول, المقال, النص, الشعري, الجزايري, المعا...","[يتناول, المقال, النص, الشعري, الجزايري, المعا...","[يتناو, المقا, الن, الشعر, الجزاير, المعاص, با...",يتناو المقا الن الشعر الجزاير المعاص بالدراسه ...,1,3,"[يتناو, المقا, الن, الشعر, الجزاير, المعاص, با...","(2.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.23121944731445102, 0.3292976696446195, 0.0,..."
3,تتناول هذه الدراسة ظاهرة الإيقاع العروضي في إل...,0,تتناول هذه الدراسه ظاهره الايقاع العروضي في ال...,تتناول هذه الدراسه ظاهره الايقاع العروضي في ال...,"[تتناول, هذه, الدراسه, ظاهره, الايقاع, العروضي...","[تتناول, الدراسه, ظاهره, الايقاع, العروضي, الي...","[تتناو, الدراس, ظاهر, الايقا, العروض, الياذ, ا...",تتناو الدراس ظاهر الايقا العروض الياذ الجزاي ،...,6,4,"[تتناو, الدراس, ظاهر, الايقا, العروض, الياذ, ا...","(3.0, 2.0, 1.0, 0.0, 4.0, 1.0, 0.0, 0.0, 0.0, ...","(0.34682917097167654, 0.3292976696446195, 0.71..."
4,يعدُّ النصُّ في النَّقد العربي الحديث محور الع...,0,يعدُّ النصُّ في النَّقد العربي الحديث محور الع...,يعد النص في النقد العربي الحديث محور العمليه ا...,"[يعد, النص, في, النقد, العربي, الحديث, محور, ا...","[يعد, النص, النقد, العربي, الحديث, محور, العمل...","[يعد, الن, النق, العرب, الحدي, محو, العملي, ال...",يعد الن النق العرب الحدي محو العملي النقدي ومن...,3,2,"[يعد, الن, النق, العرب, الحدي, محو, العملي, ال...","(1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, ...","(0.11560972365722551, 0.0, 0.0, 0.0, 0.0, 0.77..."
5,يدرسُ هذا البحثُ بلاغةَ التمثيل في القرآن الكر...,0,يدرسُ هذا البحثُ بلاغهَ التمثيل في القران الكر...,يدرس هذا البحث بلاغه التمثيل في القران الكريم ...,"[يدرس, هذا, البحث, بلاغه, التمثيل, في, القران,...","[يدرس, البحث, بلاغه, التمثيل, القران, الكريم, ...","[يدر, البح, بلاغ, التمثي, القرا, الكري, خلا, ك...",يدر البح بلاغ التمثي القرا الكري خلا كتا الكشا...,4,1,"[يدر, البح, بلاغ, التمثي, القرا, الكري, خلا, ك...","(1.0, 3.0, 0.0, 3.0, 1.0, 1.0, 0.0, 1.0, 0.0, ...","(0.11560972365722551, 0.4939465044669292, 0.0,..."
6,قرينة السّياق؛ تعدُّ من الدّلالات؛ التي يعوّل ...,0,قرينه السّياق؛ تعدُّ من الدّلالات؛ التي يعوّل ...,قرينه السياق؛ تعد من الدلالات؛ التي يعول عليها...,"[قرينه, السياق؛, تعد, من, الدلالات؛, التي, يعو...","[قرينه, السياق؛, تعد, الدلالات؛, يعول, عليها, ...","[قرين, السياق, تعد, الدلالات, يعو, عليه, الليس...",قرين السياق تعد الدلالات يعو عليه الليسانيو اك...,4,2,"[قرين, السياق, تعد, الدلالات, يعو, عليه, الليس...","(2.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ...","(0.23121944731445102, 0.16464883482230974, 0.0..."
7,الاعراب من العلوم الجليلة التي خصّت بها العرب،...,0,الاعراب من العلوم الجليله التي خصّت بها العرب،...,الاعراب من العلوم الجليله التي خصت بها العرب، ...,"[الاعراب, من, العلوم, ا